# A3. Synapse banks and slots


Point-process mechanisms are different from distributed ion channels. A sodium channel density inserted twice into the same compartment should normally collapse to one compartment-local conductance density. A bank of AMPA contacts inserted at the same dendritic location should not collapse, because each contact may have independent kinetic state, short-term plasticity state, weight history, and event timing.

## Physical compartments versus local slots

A {py:class}`~dendra.models.slice.Slice` identifies physical compartments:

```python
dend_mid = cell.slice("dend", loc=0.5)
```

A {py:class}`~dendra.models.slice.SynapseSlots` object identifies local point-process slots inside one mechanism:

```python
dend_mid.insert(exp2syn, alias="exc", copies=30)
slots = dend_mid.slots(cell.mech.exc)

slots.local_index  # local slot IDs: 0..29
slots.flat_index   # parent population-flat compartment IDs
```

Many `local_index` values may map to the same `flat_index`. Those slots read the same membrane voltage and scatter-add their current into the same compartment, but they keep independent mechanism state tensors.

## Connecting to slots

Use the slot API when the target mechanism may have duplicate physical keys:

```python
net.connect_one_to_one_slots(
    pre[:30],
    slots,
    threshold=0.0,
    weight=0.05,
    delay=1.0,
)
```

or pass the slot object to the usual connection API:

```python
net.connect_one_to_one(pre[:30], slots, threshold=0.0, weight=0.05, delay=1.0)
```

Connecting a banked mechanism through a physical slice is ambiguous when multiple slots share a compartment. Dendra raises an error rather than selecting an arbitrary slot.

## Scalar collapse versus banked equivalence

This is usually **not** equivalent:

```text
many biological contacts -> one scalar synapse state
```

because one event can change state shared by all contacts.

This is equivalent to many point processes when the mechanism state is elementwise:

```text
many biological contacts -> one vectorized mechanism -> one local slot per contact
```

The model still has one independent state element per contact; it simply avoids one Python mechanism object per contact.

## Once-per-step recovery

Mechanisms that implement `net_receive(...)` should distinguish event-triggered increments from continuous or per-step recovery. If several NetCon groups can target the same bank in one timestep, recovery/decay that conceptually occurs once per timestep should be guarded or represented as ODE state so it is not applied once per incoming NetCon call.